In [1]:
import pandas as pd
import ast
geneagent_path = "AML_GenegentAnnotationWithoutContext.csv"
our_path = "AML Validation.csv"
output_path = "AML Validation V1.csv"
geneagent_df = pd.read_csv(geneagent_path)
our_df = pd.read_csv(our_path)
def normalize_geneagent(genes):
    return "|".join(sorted(genes.split()))

def normalize_ours(genes):
    gene_list = ast.literal_eval(genes)
    return "|".join(sorted(gene_list))


geneagent_df["gene_key"] = geneagent_df["genes"].apply(normalize_geneagent)
our_df["gene_key"] = our_df["Genes"].apply(normalize_ours)

gene_map = dict(zip(geneagent_df["gene_key"], geneagent_df["name"]))

our_df["GeneSet_Name"] = our_df["gene_key"].map(gene_map)
missing = our_df["GeneSet_Name"].isna().sum()
print("Rows without match:", missing)
our_df = our_df.drop(columns=["gene_key"])
our_df.to_csv(output_path, index=False)
print("Saved:", output_path)

Rows without match: 0
Saved: AML Validation V1.csv


In [2]:
import pandas as pd
import ast

our_file = "AML Validation V1.csv"
geneagent_file = "AML_GenegentAnnotationWithoutContext.csv"
hu_file = "AML_HuAnnotationwithoutContext.csv"

output_file = "AMLWithoutContext_Combined_Annotations.csv"

our = pd.read_csv(our_file)
geneagent = pd.read_csv(geneagent_file)
hu = pd.read_csv(hu_file)


# ---- OUR ----
def normalize(g):
    try:
        return ", ".join(sorted(ast.literal_eval(g)))
    except:
        return g

our["Genes"] = our["Genes"].apply(normalize)

our = our.rename(columns={
    "GeneSet_Name": "Geneset",
    "Final_Process": "Our Annotation"
})

our = our[["Geneset", "Genes", "Our Annotation"]]


# ---- GeneAgent ----
geneagent = geneagent.rename(columns={
    "name": "Geneset",
    "final_process_name": "GeneAgent Annotation"
})

geneagent = geneagent[["Geneset", "GeneAgent Annotation"]]


# ---- Hu et al. ----
hu = hu.rename(columns={
    "gene_set": "Geneset",
    "process_name": "Hu Annotation"
})

hu = hu[["Geneset", "Hu Annotation"]]


# ---- Merge ----
merged = our.merge(geneagent, on="Geneset", how="left")
merged = merged.merge(hu, on="Geneset", how="left")

merged = merged[
    ["Geneset", "Genes", "Hu Annotation", "GeneAgent Annotation", "Our Annotation"]
]


merged.to_csv(output_file, index=False)

print("Saved:", output_file)
print("Rows:", len(merged))

Saved: AMLWithoutContext_Combined_Annotations.csv
Rows: 64


In [3]:
import pandas as pd
import ast

our_file = "AML Validation V1.csv"
geneagent_file = "AML_GeneagentAnnotationWithContext.csv"
hu_file = "AML_HuAnnotationwithcontext.csv"

output_file = "AMLWithContext_Combined_Annotations.csv"

our = pd.read_csv(our_file)
geneagent = pd.read_csv(geneagent_file)
hu = pd.read_csv(hu_file)


# ---- OUR ----
def normalize(g):
    try:
        return ", ".join(sorted(ast.literal_eval(g)))
    except:
        return g

our["Genes"] = our["Genes"].apply(normalize)

our = our.rename(columns={
    "GeneSet_Name": "Geneset",
    "Final_Process": "Our Annotation"
})

our = our[["Geneset", "Genes", "Our Annotation"]]


# ---- GeneAgent ----
geneagent = geneagent.rename(columns={
    "name": "Geneset",
    "final_process_name": "GeneAgent Annotation"
})

geneagent = geneagent[["Geneset", "GeneAgent Annotation"]]


# ---- Hu et al. ----
hu = hu.rename(columns={
    "gene_set": "Geneset",
    "process_name": "Hu Annotation"
})

hu = hu[["Geneset", "Hu Annotation"]]


# ---- Merge ----
merged = our.merge(geneagent, on="Geneset", how="left")
merged = merged.merge(hu, on="Geneset", how="left")

merged = merged[
    ["Geneset", "Genes", "Hu Annotation", "GeneAgent Annotation", "Our Annotation"]
]


merged.to_csv(output_file, index=False)

print("Saved:", output_file)
print("Rows:", len(merged))

Saved: AMLWithContext_Combined_Annotations.csv
Rows: 64


In [4]:
import pandas as pd
import re
from pathlib import Path
import os
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [5]:
def run_ladder_validation(df,
                          out_dir="Intermediate Files",
                          csv_name="Semantic_results_general_onlyV1.csv",
                          max_length=512,
                          device=None,
                          run_name=None,
                          add_timestamp=False):

    import os
    import numpy as np
    import pandas as pd
    import torch
    from tqdm import tqdm
    from transformers import AutoTokenizer, AutoModel
    from sklearn.metrics.pairwise import cosine_similarity

    MODELS = {
        'BioLORD-2023': 'FremyCompany/BioLORD-2023',
        'MedCPT': 'ncbi/MedCPT-Query-Encoder'
    }

    METHOD_NAMES = {
        'Our': 'LADDER',
        'Hu': 'Hu et al',
        'GeneAgent': 'GeneAgent'
    }

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    os.makedirs(out_dir, exist_ok=True)

    prefix = f"{run_name}_" if run_name else ""
    if add_timestamp:
        from datetime import datetime
        ts = datetime.now().strftime("%Y%m%dT%H%M%S")
        prefix = f"{prefix}{ts}_" if prefix else f"{ts}_"

    csv_final = os.path.join(out_dir, f"{prefix}{csv_name}" if prefix else csv_name)

    def load_model(model_id):
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        model = AutoModel.from_pretrained(model_id)
        model.to(device)
        model.eval()
        return tokenizer, model

    def get_embedding(text, tokenizer, model):
        if not isinstance(text, str) or len(text.strip()) == 0:
            return np.zeros(model.config.hidden_size, dtype=float)

        inputs = tokenizer(text,
                           return_tensors="pt",
                           truncation=True,
                           max_length=max_length,
                           padding=True)

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            emb = outputs.last_hidden_state[:, 0, :].cpu().numpy().flatten()

        return emb

    def validate_with_model(df_local, display_name, model_id):
        tokenizer, model = load_model(model_id)

        rows = []

        for _, row in tqdm(df_local.iterrows(), total=len(df_local), desc=display_name):

            desc_emb = get_embedding(row.get('MSigDB_Brief_Description', ''), tokenizer, model)
            hu_emb = get_embedding(row.get('Hu Annotation', ''), tokenizer, model)
            ga_emb = get_embedding(row.get('GeneAgent Annotation', ''), tokenizer, model)
            our_emb = get_embedding(row.get('Our Annotation', ''), tokenizer, model)

            eps = 1e-12

            hu_sim = cosine_similarity(desc_emb.reshape(1, -1)+eps,
                                       hu_emb.reshape(1, -1)+eps)[0][0]

            ga_sim = cosine_similarity(desc_emb.reshape(1, -1)+eps,
                                       ga_emb.reshape(1, -1)+eps)[0][0]

            our_sim = cosine_similarity(desc_emb.reshape(1, -1)+eps,
                                        our_emb.reshape(1, -1)+eps)[0][0]

            scores = {'Our': our_sim, 'Hu': hu_sim, 'GeneAgent': ga_sim}
            winner = max(scores, key=scores.get)

            rows.append({
                "Geneset": row.get("Geneset"),
                "LADDER_Similarity": our_sim,
                "Hu_Similarity": hu_sim,
                "GeneAgent_Similarity": ga_sim,
                "Winner": METHOD_NAMES[winner],
                "Model": display_name
            })

        del model, tokenizer
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return pd.DataFrame(rows)

    results = []

    for name, path in MODELS.items():
        try:
            res = validate_with_model(df, name, path)
            results.append(res)
        except Exception as e:
            print(f"Model failed: {name} -> {e}")

    results_df = pd.concat(results, ignore_index=True)

    results_df.to_csv(csv_final, index=False)

    print(f"Saved results: {csv_final}")

    return results_df

In [6]:
combined_path = "AMLWithoutContext_Combined_Annotations.csv"
combined = pd.read_csv(combined_path)
combined_first6 = combined.iloc[:, :5]
annotations_df = combined_first6
msigdb_df = pd.read_csv('AML_msigdb_descriptions.csv')
df = annotations_df.merge(msigdb_df, on='Geneset')
print(f"Total genesets: {len(df)}\n")
results_df = run_ladder_validation(df, run_name="AMLWITHOUTCONTEXT", add_timestamp=False)

Total genesets: 64



PubMedBERT: 100%|█████████████████████████████| 64/64 [00:08<00:00,  7.55it/s]

Saved results: Intermediate Files/AMLWITHOUTCONTEXT_Semantic_results_general_onlyV1.csv


In [7]:
combined_path = "AMLWithContext_Combined_Annotations.csv"
combined = pd.read_csv(combined_path)
combined_first6 = combined.iloc[:, :5]
annotations_df = combined_first6
msigdb_df = pd.read_csv('AML_msigdb_descriptions.csv')
df = annotations_df.merge(msigdb_df, on='Geneset')
print(f"Total genesets: {len(df)}\n")
results_df = run_ladder_validation(df, run_name="AMLWITHCONTEXT", add_timestamp=False)

Total genesets: 64



PubMedBERT: 100%|█████████████████████████████| 64/64 [00:09<00:00,  6.60it/s]

Saved results: Intermediate Files/AMLWITHCONTEXT_Semantic_results_general_onlyV1.csv


In [5]:
import pandas as pd

df = pd.read_csv("AMLWithContext_Combined_Annotations.csv")

# Count genes per geneset (Genes column is comma-separated)
df["GeneCount"] = df["Genes"].apply(lambda x: len(str(x).split(",")))

# Summary statistics
num_genesets = len(df)
min_genes = df["GeneCount"].min()
max_genes = df["GeneCount"].max()
avg_genes = df["GeneCount"].mean()

print("=== AML Annotation Summary ===")
print(f"Number of genesets:        {num_genesets}")
print(f"Minimum genes per set:     {min_genes}")
print(f"Maximum genes per set:     {max_genes}")
print(f"Average genes per set:     {avg_genes:.2f}")


=== AML Annotation Summary ===
Number of genesets:        64
Minimum genes per set:     5
Maximum genes per set:     411
Average genes per set:     72.69


In [6]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import pandas as pd
import glob
import os
from rouge_score import rouge_scorer
from tqdm import tqdm

combined_paths = [
    "AMLWithoutContext_Combined_Annotations.csv",
    "AMLWithContext_Combined_Annotations.csv",
]

msigdb_map = {
       "AML": "AML_msigdb_descriptions.csv"
}

output_dir = "Intermediate Files/AMLrouge_aml_results"
os.makedirs(output_dir, exist_ok=True)
# -----------------------------------------------------

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

def choose_msigdb_path(csv_path):
    lname = os.path.basename(csv_path).lower()
    for key, path in msigdb_map.items():
        if key.lower() in lname:
            return path
    raise ValueError(f"Could not find MSigDB mapping for file {csv_path}. Edit msigdb_map if needed.")

def safe_text(x):
    if pd.isna(x):
        return ""
    return str(x)

all_rows = []
summary = []

print("="*80)
print("Computing ROUGE-1, ROUGE-2, and ROUGE-L metrics")
print("="*80)

for combined_path in combined_paths:
    print(f"\nProcessing: {combined_path}")
    df = pd.read_csv(combined_path, dtype=str)
    msigdb_path = choose_msigdb_path(combined_path)
    msig = pd.read_csv(msigdb_path, dtype=str)

    msig = msig.rename(columns={c: c.strip() for c in msig.columns})
    df = df.rename(columns={c: c.strip() for c in df.columns})

    # Join on Geneset column
    if 'Geneset' not in df.columns:
        raise ValueError(f"'Geneset' column not found in {combined_path}")
    merged = df.merge(msig[['Geneset','MSigDB_Brief_Description']], on='Geneset', how='left')

    per_file_rows = []
    # iterate rows
    for _, row in tqdm(merged.iterrows(), total=len(merged), desc=os.path.basename(combined_path)):
        geneset = row['Geneset']
        ref = safe_text(row.get('MSigDB_Brief_Description', ""))
        # candidate annotations
        our = safe_text(row.get('Our Annotation', ""))
        hu = safe_text(row.get('Hu Annotation', ""))
        ga = safe_text(row.get('GeneAgent Annotation', ""))

        # compute all rouge metrics (rouge1, rouge2, rougeL)
        def score_pair(reference, hypothesis):
            if len(reference.strip()) == 0 and len(hypothesis.strip()) == 0:
                return {
                    'rouge1': {'precision': 1.0, 'recall': 1.0, 'fmeasure': 1.0},
                    'rouge2': {'precision': 1.0, 'recall': 1.0, 'fmeasure': 1.0},
                    'rougeL': {'precision': 1.0, 'recall': 1.0, 'fmeasure': 1.0}
                }
            if len(reference.strip()) == 0:
                return {
                    'rouge1': {'precision': 0.0, 'recall': 0.0, 'fmeasure': 0.0},
                    'rouge2': {'precision': 0.0, 'recall': 0.0, 'fmeasure': 0.0},
                    'rougeL': {'precision': 0.0, 'recall': 0.0, 'fmeasure': 0.0}
                }
            
            scores = scorer.score(reference, hypothesis)
            result = {}
            for rouge_type in ['rouge1', 'rouge2', 'rougeL']:
                s = scores[rouge_type]
                result[rouge_type] = {
                    'precision': s.precision,
                    'recall': s.recall,
                    'fmeasure': s.fmeasure
                }
            return result

        our_scores = score_pair(ref, our)
        hu_scores = score_pair(ref, hu)
        ga_scores = score_pair(ref, ga)

        rouge_types = ['rouge1', 'rouge2', 'rougeL']
        best_methods = {}
        best_f1s = {}
        
        for rouge_type in rouge_types:
            method_scores = {
                'Our': our_scores[rouge_type]['fmeasure'],
                'Hu': hu_scores[rouge_type]['fmeasure'],
                'GeneAgent': ga_scores[rouge_type]['fmeasure']
            }
            best_method = max(method_scores, key=method_scores.get)
            best_methods[rouge_type] = best_method
            best_f1s[rouge_type] = method_scores[best_method]

        row_dict = {
            'source_file': os.path.basename(combined_path),
            'Geneset': geneset,
            'MSigDB_Brief_Description': ref,
            'Our': our,
            'Hu': hu,
            'GeneAgent': ga,
        }
        
        for rouge_type in rouge_types:
            row_dict[f'Our_{rouge_type}_p'] = our_scores[rouge_type]['precision']
            row_dict[f'Our_{rouge_type}_r'] = our_scores[rouge_type]['recall']
            row_dict[f'Our_{rouge_type}_f'] = our_scores[rouge_type]['fmeasure']
            row_dict[f'Hu_{rouge_type}_p'] = hu_scores[rouge_type]['precision']
            row_dict[f'Hu_{rouge_type}_r'] = hu_scores[rouge_type]['recall']
            row_dict[f'Hu_{rouge_type}_f'] = hu_scores[rouge_type]['fmeasure']
            row_dict[f'GeneAgent_{rouge_type}_p'] = ga_scores[rouge_type]['precision']
            row_dict[f'GeneAgent_{rouge_type}_r'] = ga_scores[rouge_type]['recall']
            row_dict[f'GeneAgent_{rouge_type}_f'] = ga_scores[rouge_type]['fmeasure']
            # Best method for this metric
            row_dict[f'Best_method_by_{rouge_type}_F1'] = best_methods[rouge_type]
            row_dict[f'Best_{rouge_type}_F1'] = best_f1s[rouge_type]
        
        per_file_rows.append(row_dict)

    per_file_df = pd.DataFrame(per_file_rows)
    per_file_out = os.path.join(output_dir, os.path.basename(combined_path).replace('.csv','') + "_all_rouge_per_geneset.csv")
    per_file_df.to_csv(per_file_out, index=False)
    print(f"✓ Saved per-geneset results to: {per_file_out}")

    # Summary counts for each ROUGE metric
    summary_row = {
        'source_file': os.path.basename(combined_path),
        'n_genesets': len(per_file_df),
    }
    
    for rouge_type in rouge_types:
        counts = per_file_df[f'Best_method_by_{rouge_type}_F1'].value_counts().to_dict()
        summary_row[f'{rouge_type}_Our_best_count'] = counts.get('Our', 0)
        summary_row[f'{rouge_type}_Hu_best_count'] = counts.get('Hu', 0)
        summary_row[f'{rouge_type}_GeneAgent_best_count'] = counts.get('GeneAgent', 0)
        summary_row[f'{rouge_type}_Our_best_fraction'] = counts.get('Our', 0)/len(per_file_df) if len(per_file_df)>0 else 0
        summary_row[f'{rouge_type}_Hu_best_fraction'] = counts.get('Hu', 0)/len(per_file_df) if len(per_file_df)>0 else 0
        summary_row[f'{rouge_type}_GeneAgent_best_fraction'] = counts.get('GeneAgent', 0)/len(per_file_df) if len(per_file_df)>0 else 0
        
        # Top 10 genesets by F1 for each method
        summary_row[f'{rouge_type}_top_10_our_by_f1'] = ';'.join(per_file_df.sort_values(f'Our_{rouge_type}_f', ascending=False).head(10)['Geneset'].tolist())
        summary_row[f'{rouge_type}_top_10_hu_by_f1'] = ';'.join(per_file_df.sort_values(f'Hu_{rouge_type}_f', ascending=False).head(10)['Geneset'].tolist())
        summary_row[f'{rouge_type}_top_10_ga_by_f1'] = ';'.join(per_file_df.sort_values(f'GeneAgent_{rouge_type}_f', ascending=False).head(10)['Geneset'].tolist())
        
        # Average scores
        summary_row[f'{rouge_type}_Our_avg_f1'] = per_file_df[f'Our_{rouge_type}_f'].mean()
        summary_row[f'{rouge_type}_Hu_avg_f1'] = per_file_df[f'Hu_{rouge_type}_f'].mean()
        summary_row[f'{rouge_type}_GeneAgent_avg_f1'] = per_file_df[f'GeneAgent_{rouge_type}_f'].mean()
    
    summary.append(summary_row)


    all_rows.extend(per_file_rows)


combined_out = os.path.join(output_dir, "all_combined_all_rouge_per_geneset.csv")
pd.DataFrame(all_rows).to_csv(combined_out, index=False)
print(f"\n✓ Saved combined per-geneset table to: {combined_out}")


summary_df = pd.DataFrame(summary)
summary_out = os.path.join(output_dir, "all_rouge_summary_by_file.csv")
summary_df.to_csv(summary_out, index=False)
print(f"✓ Saved summary table to: {summary_out}")

report_path = os.path.join(output_dir, "summary_report.txt")
with open(report_path, 'w') as f:
    f.write("="*80 + "\n")
    f.write("ROUGE-1, ROUGE-2, and ROUGE-L EVALUATION SUMMARY\n")
    f.write("="*80 + "\n\n")
    
    total_genesets = summary_df['n_genesets'].sum()
    f.write(f"Total genesets evaluated: {total_genesets}\n")
    f.write(f"Number of datasets: {len(summary_df)}\n\n")
    
    for rouge_type in ['rouge1', 'rouge2', 'rougeL']:
        f.write(f"\n{rouge_type.upper()} METRICS\n")
        f.write("-"*80 + "\n")
        f.write("\nOverall Win Counts:\n")
        for method in ['Our', 'Hu', 'GeneAgent']:
            count = summary_df[f'{rouge_type}_{method}_best_count'].sum()
            pct = (count / total_genesets) * 100
            f.write(f"  {method}: {count} ({pct:.2f}%)\n")
        
        f.write("\nAverage F1 Scores (across all datasets):\n")
        for method in ['Our', 'Hu', 'GeneAgent']:
            avg_f1 = summary_df[f'{rouge_type}_{method}_avg_f1'].mean()
            f.write(f"  {method}: {avg_f1:.4f}\n")
    
    f.write("\n" + "="*80 + "\n")
    f.write("DETAILED BREAKDOWN BY DATASET\n")
    f.write("="*80 + "\n")
    
    for idx, row in summary_df.iterrows():
        f.write(f"\n{row['source_file']}\n")
        f.write("-"*80 + "\n")
        f.write(f"Number of genesets: {row['n_genesets']}\n\n")
        
        for rouge_type in ['rouge1', 'rouge2', 'rougeL']:
            f.write(f"  {rouge_type.upper()}:\n")
            for method in ['Our', 'Hu', 'GeneAgent']:
                count = row[f'{rouge_type}_{method}_best_count']
                fraction = row[f'{rouge_type}_{method}_best_fraction']
                avg_f1 = row[f'{rouge_type}_{method}_avg_f1']
                f.write(f"    {method}: {int(count)} wins ({fraction*100:.1f}%), Avg F1={avg_f1:.4f}\n")
            f.write("\n")

print(f"✓ Saved summary report to: {report_path}")

print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"\nTotal genesets processed: {len(all_rows)}")
print(f"\nOutput directory: {output_dir}")
print(f"Files generated:")
print(f"  - {len(combined_paths)} per-file CSV files")
print(f"  - 1 combined CSV file: all_combined_all_rouge_per_geneset.csv")
print(f"  - 1 summary CSV file: all_rouge_summary_by_file.csv")
print(f"  - 1 text report: summary_report.txt")

print("\n" + "="*80)
print("All ROUGE metrics computation complete!")
print("="*80)

Computing ROUGE-1, ROUGE-2, and ROUGE-L metrics

Processing: AMLWithoutContext_Combined_Annotations.csv


AMLWithoutContext_Combined_Annotations.csv: 100%|███████████████| 64/64 [00:00<00:00, 1487.27it/s]


✓ Saved per-geneset results to: Intermediate Files/AMLrouge_aml_results/AMLWithoutContext_Combined_Annotations_all_rouge_per_geneset.csv

Processing: AMLWithContext_Combined_Annotations.csv


AMLWithContext_Combined_Annotations.csv: 100%|██████████████████| 64/64 [00:00<00:00, 2274.59it/s]

✓ Saved per-geneset results to: Intermediate Files/AMLrouge_aml_results/AMLWithContext_Combined_Annotations_all_rouge_per_geneset.csv

✓ Saved combined per-geneset table to: Intermediate Files/AMLrouge_aml_results/all_combined_all_rouge_per_geneset.csv
✓ Saved summary table to: Intermediate Files/AMLrouge_aml_results/all_rouge_summary_by_file.csv
✓ Saved summary report to: Intermediate Files/AMLrouge_aml_results/summary_report.txt

SUMMARY

Total genesets processed: 128

Output directory: Intermediate Files/AMLrouge_aml_results
Files generated:
  - 2 per-file CSV files
  - 1 combined CSV file: all_combined_all_rouge_per_geneset.csv
  - 1 summary CSV file: all_rouge_summary_by_file.csv
  - 1 text report: summary_report.txt

All ROUGE metrics computation complete!


In [7]:
import pandas as pd
import os

# ---------------- paths ----------------
annotation_file = "AML Validation V1.csv"

target_files = [
"Intermediate Files/AMLrouge_aml_results/AMLWithContext_Combined_Annotations_all_rouge_per_geneset.csv",
"Intermediate Files/AMLrouge_aml_results/AMLWithoutContext_Combined_Annotations_all_rouge_per_geneset.csv",
"Intermediate Files/AMLWITHCONTEXT_Semantic_results_general_only.csv",
"Intermediate Files/AMLWITHOUTCONTEXT_Semantic_results_general_only.csv"
]

# ---------------- load confidence ----------------
conf = pd.read_csv(annotation_file, dtype=str)[["GeneSet_Name","Final_Confidence"]]
conf = conf.rename(columns={"GeneSet_Name":"Geneset"})

# ---------------- process each file ----------------
for path in target_files:

    df = pd.read_csv(path, dtype=str)

    merged = df.merge(conf, on="Geneset", how="left")

    out_path = path.replace(".csv","_withConfidence.csv")
    merged.to_csv(out_path, index=False)

    missing = merged["Final_Confidence"].isna().sum()

    print(f"\nFile processed: {os.path.basename(path)}")
    print(f"Rows: {len(merged)}")
    print(f"Missing confidence: {missing}")
    print(f"Saved: {out_path}")


File processed: AMLWithContext_Combined_Annotations_all_rouge_per_geneset.csv
Rows: 64
Missing confidence: 0
Saved: Intermediate Files/AMLrouge_aml_results/AMLWithContext_Combined_Annotations_all_rouge_per_geneset_withConfidence.csv

File processed: AMLWithoutContext_Combined_Annotations_all_rouge_per_geneset.csv
Rows: 64
Missing confidence: 0
Saved: Intermediate Files/AMLrouge_aml_results/AMLWithoutContext_Combined_Annotations_all_rouge_per_geneset_withConfidence.csv

File processed: AMLWITHCONTEXT_Semantic_results_general_only.csv
Rows: 320
Missing confidence: 0
Saved: Intermediate Files/AMLWITHCONTEXT_Semantic_results_general_only_withConfidence.csv

File processed: AMLWITHOUTCONTEXT_Semantic_results_general_only.csv
Rows: 320
Missing confidence: 0
Saved: Intermediate Files/AMLWITHOUTCONTEXT_Semantic_results_general_only_withConfidence.csv
